# 04 - Benchmark GARCH-t

Este notebook contiene el codigo completo del benchmark y guarda sus predicciones en `results/predictions/`.


## Definicion del benchmark

Esta celda define el ajuste rolling GARCH(1,1) con innovaciones t de Student.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from arch import arch_model
from scipy.optimize import minimize
from scipy.stats import norm, t as t_dist
from tqdm.auto import tqdm


warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
PREDICTIONS = RESULTS / "predictions"
PREDICTIONS.mkdir(parents=True, exist_ok=True)

ALPHA = 0.99
WINDOW = 900
EVAL_START = "2015-01-01"
EVAL_END = "2024-12-31"


def load_dataset() -> pd.DataFrame:
    return pd.read_pickle(DATA / "dataset_tfg.pkl").sort_index()


def run_historical_simulation(dataset: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for current_date in tqdm(dataset.loc[EVAL_START:EVAL_END].index, desc="HS rolling", dynamic_ncols=True):
        train_df = dataset.loc[: current_date - pd.Timedelta(days=1)].tail(WINDOW)
        loss_train = train_df["target"].dropna().to_numpy()
        if len(loss_train) < 250:
            continue
        var_pred = float(np.quantile(loss_train, ALPHA))
        rows.append((current_date, var_pred, float(dataset.loc[current_date, "target"])))

    out = pd.DataFrame(rows, columns=["date", "VaR_HS", "loss_real"]).set_index("date")
    out["viol"] = (out["loss_real"] > out["VaR_HS"]).astype(int)
    out["year"] = out.index.year
    out.to_csv(PREDICTIONS / "hs_var_predictions.csv")
    return out


def run_garch(dataset: pd.DataFrame, dist: str) -> pd.DataFrame:
    var_col = "VaR_GARCH" if dist == "t" else "VaR_GARCH_n"
    rows = []
    current_model = None
    for current_date in tqdm(dataset.loc[EVAL_START:EVAL_END].index, desc=f"GARCH-{dist} rolling", dynamic_ncols=True):
        train_df = dataset.loc[: current_date - pd.Timedelta(days=1)].tail(WINDOW)
        r_train = (train_df["rp_lag_0"].dropna() * 100.0).astype(float)
        if len(r_train) >= 250:
            am = arch_model(r_train, mean="Constant", vol="GARCH", p=1, q=1, dist=("t" if dist == "t" else "normal"))
            try:
                current_model = am.fit(disp="off")
            except Exception:
                pass

        if current_model is None:
            continue

        forecast = current_model.forecast(horizon=1, reindex=False)
        mu = float(forecast.mean.iloc[-1, 0])
        sigma = float(np.sqrt(forecast.variance.iloc[-1, 0]))
        if dist == "t":
            nu = float(current_model.params["nu"])
            q = t_dist.ppf(1.0 - ALPHA, df=nu)
        else:
            q = norm.ppf(1.0 - ALPHA)
        var_pred = -(mu + sigma * q) / 100.0
        rows.append((current_date, float(var_pred), float(dataset.loc[current_date, "target"])))

    out = pd.DataFrame(rows, columns=["date", var_col, "loss_real"]).set_index("date")
    out["viol"] = (out["loss_real"] > out[var_col]).astype(int)
    out["year"] = out.index.year
    filename = "garch_t_var_predictions.csv" if dist == "t" else "garch_n_var_predictions.csv"
    out.to_csv(PREDICTIONS / filename)
    return out


def caviar_seq(params, r):
    b0, b1, b2, b3 = params
    var = np.empty(len(r))
    var[0] = np.quantile(r[:100], ALPHA)
    for i in range(1, len(r)):
        lag = r[i - 1]
        var[i] = b0 + b1 * var[i - 1] + b2 * max(lag, 0.0) + b3 * min(lag, 0.0)
    return var


def caviar_loss(params, r):
    var = caviar_seq(params, r)
    e = r - var
    return float(np.mean(np.where(e >= 0.0, ALPHA * e, (ALPHA - 1.0) * e)))


def fit_caviar(r_train):
    result = minimize(
        caviar_loss,
        x0=[0.01, 0.9, 0.1, 0.1],
        args=(r_train,),
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-6, "fatol": 1e-7, "adaptive": True},
    )
    return result.x


def predict_caviar(params, r_train):
    b0, b1, b2, b3 = params
    var_t = caviar_seq(params, r_train)[-1]
    lag = r_train[-1]
    return float(b0 + b1 * var_t + b2 * max(lag, 0.0) + b3 * min(lag, 0.0))


def run_caviar_as(dataset: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for current_date in tqdm(dataset.loc[EVAL_START:EVAL_END].index, desc="CAViaR-AS rolling", dynamic_ncols=True):
        train_df = dataset.loc[: current_date - pd.Timedelta(days=1)].tail(WINDOW)
        r_train = train_df["target"].dropna().to_numpy()
        if len(r_train) < 250:
            continue
        params = fit_caviar(r_train)
        var_pred = predict_caviar(params, r_train)
        rows.append((current_date, var_pred, float(dataset.loc[current_date, "target"])))

    out = pd.DataFrame(rows, columns=["date", "VaR_CAViaR", "loss_real"]).set_index("date")
    out["viol"] = (out["loss_real"] > out["VaR_CAViaR"]).astype(int)
    out["year"] = out.index.year
    out.to_csv(PREDICTIONS / "caviar_var_predictions.csv")
    return out


def main() -> None:
    dataset = load_dataset()
    run_historical_simulation(dataset)
    run_garch(dataset, dist="t")
    run_garch(dataset, dist="normal")
    run_caviar_as(dataset)
    print("Saved benchmark predictions in", PREDICTIONS)


def main_hs() -> None:
    dataset = load_dataset()
    run_historical_simulation(dataset)
    print("Saved Historical Simulation predictions in", PREDICTIONS)


def main_garch_t() -> None:
    dataset = load_dataset()
    run_garch(dataset, dist="t")
    print("Saved GARCH-t predictions in", PREDICTIONS)


def main_garch_normal() -> None:
    dataset = load_dataset()
    run_garch(dataset, dist="normal")
    print("Saved GARCH-Normal predictions in", PREDICTIONS)


def main_caviar_as() -> None:
    dataset = load_dataset()
    run_caviar_as(dataset)
    print("Saved CAViaR-AS predictions in", PREDICTIONS)


## Ejecucion

Esta celda calcula las predicciones de VaR GARCH-t y las guarda en `results/predictions/`.


In [ ]:
main_garch_t()


## Comprobacion de outputs

Esta celda carga las predicciones guardadas y muestra un resumen rapido.


In [ ]:
df = pd.read_csv(PREDICTIONS / "garch_t_var_predictions.csv", index_col=0, parse_dates=True)
display(df.head())
display(df.describe())
